In [1]:
import torch
import torchvision
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torch.utils.data as dataloader
import torch.nn as nn

In [2]:
transformation_operation = transforms.Compose([transforms.ToTensor()])

In [3]:
num_classes = 10
num_channels = 1
img_size = 28
patch_size = 7
num_patches = (img_size // patch_size)**2
print(num_patches)
embedding_dim = 64
attention_heads = 4
mlp_nodes = 128

16


In [6]:
import torch.optim as optim

# 1. 准备数据
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transformation_operation)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transformation_operation)

train_loader = dataloader.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = dataloader.DataLoader(test_dataset, batch_size=1000, shuffle=False)

# 2. 初始化模型和优化器
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"正在使用 {device} 进行训练...")
model = MiniViT().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 3. 开始训练（只跑 3 个 epoch，作为演示）
epochs = 3
for epoch in range(epochs):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        if batch_idx % 200 == 0:
            print(f"Epoch: {epoch+1}/{epochs} | Batch: {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    # 测试准确率
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    print(f"===== Epoch {epoch+1} 结束，测试准确率: {100 * correct / total:.2f}% =====")

正在使用 cpu 进行训练...
Epoch: 1/3 | Batch: 0/938 | Loss: 2.3778
Epoch: 1/3 | Batch: 200/938 | Loss: 0.6061
Epoch: 1/3 | Batch: 400/938 | Loss: 0.1825
Epoch: 1/3 | Batch: 600/938 | Loss: 0.2017
Epoch: 1/3 | Batch: 800/938 | Loss: 0.1368
===== Epoch 1 结束，测试准确率: 94.27% =====
Epoch: 2/3 | Batch: 0/938 | Loss: 0.2479
Epoch: 2/3 | Batch: 200/938 | Loss: 0.0691
Epoch: 2/3 | Batch: 400/938 | Loss: 0.0983
Epoch: 2/3 | Batch: 600/938 | Loss: 0.0895
Epoch: 2/3 | Batch: 800/938 | Loss: 0.0593
===== Epoch 2 结束，测试准确率: 96.17% =====
Epoch: 3/3 | Batch: 0/938 | Loss: 0.0215
Epoch: 3/3 | Batch: 200/938 | Loss: 0.2507
Epoch: 3/3 | Batch: 400/938 | Loss: 0.1459
Epoch: 3/3 | Batch: 600/938 | Loss: 0.0634
Epoch: 3/3 | Batch: 800/938 | Loss: 0.1445
===== Epoch 3 结束，测试准确率: 96.63% =====


In [5]:
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=28, patch_size=7, in_channels=1, embed_dim=64):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, embed_dim))

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(embed_dim * mlp_ratio, embed_dim)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class MiniViT(nn.Module):
    def __init__(self, img_size=28, patch_size=7, in_channels=1, num_classes=10, embed_dim=64, depth=2, num_heads=4):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])

print("模型图纸准备完毕！")

模型图纸准备完毕！
